# Fraud Detection Model Training

This notebook trains a Random Forest classifier on the Kaggle Credit Card Fraud Detection dataset.
It applies SMOTE to handle class imbalance and saves the trained model and scaler for the API.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from imblearn.over_sampling import SMOTE

print('Libraries loaded successfully')

## 1. Load Data

In [ ]:
data_path = os.path.join('..', 'data', 'creditcard.csv')
df = pd.read_csv(data_path)

print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 2. Class Distribution

In [ ]:
class_counts = df['Class'].value_counts()
fraud_pct = class_counts[1] / len(df) * 100

print('Class Distribution:')
print(f'  Legitimate (0): {class_counts[0]:,} ({100 - fraud_pct:.2f}%)')
print(f'  Fraud      (1): {class_counts[1]:,} ({fraud_pct:.4f}%)')
print(f'  Imbalance ratio: {class_counts[0] / class_counts[1]:.1f}:1')

## 3. Feature Engineering

In [ ]:
# Derive time-based features from the Time column (seconds elapsed since first transaction)
df['hour_of_day'] = (df['Time'] % 86400) / 3600
df['day_of_week'] = (df['Time'] // 86400 % 7).astype(int)

print('Time features created:')
print(f'  hour_of_day range: {df["hour_of_day"].min():.2f} - {df["hour_of_day"].max():.2f}')
print(f'  day_of_week unique values: {sorted(df["day_of_week"].unique())}')

In [ ]:
# Scale Amount using StandardScaler
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])

# Drop original Time and Amount columns
df_processed = df.drop(columns=['Time', 'Amount'])

# Rename Amount_scaled back to Amount for the feature set
df_processed = df_processed.rename(columns={'Amount_scaled': 'Amount'})

print(f'Processed dataset shape: {df_processed.shape}')
print(f'Features: {[c for c in df_processed.columns if c != "Class"]}')

## 4. Train/Test Split

In [ ]:
feature_cols = [c for c in df_processed.columns if c != 'Class']
X = df_processed[feature_cols]
y = df_processed['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Training set: {X_train.shape[0]:,} samples')
print(f'Test set:     {X_test.shape[0]:,} samples')
print(f'Train fraud:  {y_train.sum():,} ({y_train.mean()*100:.4f}%)')
print(f'Test fraud:   {y_test.sum():,} ({y_test.mean()*100:.4f}%)')
print(f'Feature order: {list(X.columns)}')

## 5. Apply SMOTE (training set only)

In [ ]:
smote = SMOTE(random_state=42, sampling_strategy=0.1)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print('After SMOTE:')
print(f'  Training set size: {len(X_train_resampled):,}')
print(f'  Fraud samples:     {y_train_resampled.sum():,}')
print(f'  Legitimate:        {(y_train_resampled == 0).sum():,}')
print(f'  Fraud ratio:       {y_train_resampled.mean()*100:.2f}%')

## 6. Train Random Forest

In [ ]:
print('Training Random Forest classifier...')

model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    max_depth=20,
    min_samples_split=5
)

model.fit(X_train_resampled, y_train_resampled)
print('Training complete!')

## 7. Evaluate on Test Set

In [ ]:
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print('=' * 60)
print('CLASSIFICATION REPORT')
print('=' * 60)
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))

roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f'ROC-AUC Score: {roc_auc:.4f}')

print('\nConfusion Matrix:')
cm = confusion_matrix(y_test, y_pred)
print(f'  True Negatives:  {cm[0][0]:,}')
print(f'  False Positives: {cm[0][1]:,}')
print(f'  False Negatives: {cm[1][0]:,}')
print(f'  True Positives:  {cm[1][1]:,}')

## 8 & 9. Save Model and Scaler

In [ ]:
model_dir = os.path.join('..', 'model')
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, 'fraud_model.pkl')
scaler_path = os.path.join(model_dir, 'scaler.pkl')

joblib.dump(model, model_path)
joblib.dump(scaler, scaler_path)

print(f'Model saved to: {os.path.abspath(model_path)}')
print(f'Scaler saved to: {os.path.abspath(scaler_path)}')
print(f'Model file size: {os.path.getsize(model_path) / 1024 / 1024:.1f} MB')
print('Model saved successfully')

## Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print('Top 10 Most Important Features:')
print(feature_importance.head(10).to_string(index=False))